In [8]:
from io import StringIO
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [9]:
USGS_ALL_MONTH = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.csv"
EARTH_RADIUS_KM = 6371.0088



In [10]:
def download_raw_usgs(url=USGS_ALL_MONTH) -> pd.DataFrame:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))

df_raw = download_raw_usgs()
print("RAW shape:", df_raw.shape)
df_raw.to_csv("raw_all_month.csv", index=False)
print("Saved raw_all_month.csv")
df_raw.head()

RAW shape: (9968, 22)
Saved raw_all_month.csv


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,ml,23,77,0.03975,0.14,...,2026-02-21T03:42:31.097Z,"10 km S of Idyllwild, CA",earthquake,0.26,0.4400,0.118624,8.0,automatic,ci,ci
1,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,ml,11,109,0.20000,0.60,...,2026-02-21T03:39:50.585Z,"25 km N of Four Mile Road, Alaska",earthquake,4.50,3.1853,0.300000,4.0,automatic,ak,ak
2,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,ml,87,24,0.04896,0.19,...,2026-02-21T03:32:01.470Z,"11 km S of Idyllwild, CA",earthquake,0.13,0.2700,0.182024,25.0,automatic,ci,ci
3,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,md,25,259,0.71220,0.27,...,2026-02-21T03:36:40.250Z,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake,0.84,17.6800,0.052903,16.0,reviewed,pr,pr
4,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,ml,33,99,0.50000,0.60,...,2026-02-21T03:08:47.425Z,"55 km NNW of Petersville, Alaska",earthquake,4.00,3.3570,0.200000,4.0,automatic,ak,ak


In [11]:
# --- Initial Data Inspection (USGS Earthquakes) ---

print("\n--- Initial Data (Head) ---")
display(df_raw.head())

# Columns used in this project (safe even if some columns are missing)
selected_cols = ["id", "time", "latitude", "longitude", "depth", "mag", "place", "type"]
selected_cols = [c for c in selected_cols if c in df_raw.columns]
df_selected = df_raw[selected_cols].copy()

print("\n--- Initial Data (Info for Selected Features) ---")
print(df_selected.info())

print("\n--- Initial Data (Missing Values for Selected Features) ---")
print(df_selected.isnull().sum())

print("\n--- Initial Data (Basic Shape) ---")
print("Rows, Columns:", df_raw.shape)
print("Selected Columns:", df_selected.shape)

# Extra inspection (recommended for report quality)
if "type" in df_selected.columns:
    print("\n--- Event Type Counts (Top 10) ---")
    print(df_selected["type"].value_counts().head(10))

if all(c in df_selected.columns for c in ["mag", "depth"]):
    print("\n--- Summary Stats (mag & depth) ---")
    print(df_selected[["mag", "depth"]].describe())


--- Initial Data (Head) ---


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,ml,23,77,0.03975,0.14,...,2026-02-21T03:42:31.097Z,"10 km S of Idyllwild, CA",earthquake,0.26,0.4400,0.118624,8.0,automatic,ci,ci
1,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,ml,11,109,0.20000,0.60,...,2026-02-21T03:39:50.585Z,"25 km N of Four Mile Road, Alaska",earthquake,4.50,3.1853,0.300000,4.0,automatic,ak,ak
2,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,ml,87,24,0.04896,0.19,...,2026-02-21T03:32:01.470Z,"11 km S of Idyllwild, CA",earthquake,0.13,0.2700,0.182024,25.0,automatic,ci,ci
3,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,md,25,259,0.71220,0.27,...,2026-02-21T03:36:40.250Z,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake,0.84,17.6800,0.052903,16.0,reviewed,pr,pr
4,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,ml,33,99,0.50000,0.60,...,2026-02-21T03:08:47.425Z,"55 km NNW of Petersville, Alaska",earthquake,4.00,3.3570,0.200000,4.0,automatic,ak,ak



--- Initial Data (Info for Selected Features) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9968 entries, 0 to 9967
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         9968 non-null   object 
 1   time       9968 non-null   object 
 2   latitude   9968 non-null   float64
 3   longitude  9968 non-null   float64
 4   depth      9968 non-null   float64
 5   mag        9967 non-null   float64
 6   place      9968 non-null   object 
 7   type       9968 non-null   object 
dtypes: float64(4), object(4)
memory usage: 623.1+ KB
None

--- Initial Data (Missing Values for Selected Features) ---
id           0
time         0
latitude     0
longitude    0
depth        0
mag          1
place        0
type         0
dtype: int64

--- Initial Data (Basic Shape) ---
Rows, Columns: (9968, 22)
Selected Columns: (9968, 8)

--- Event Type Counts (Top 10) ---
type
earthquake          9819
explosion             69
quarry bl

In [12]:
# --- Column Selection (keep only relevant fields) ---

keep_cols = ["id", "time", "latitude", "longitude", "depth", "mag", "place", "type"]
keep_cols = [c for c in keep_cols if c in df_raw.columns]  # safe if any column missing

df_cols = df_raw[keep_cols].copy()

print("\n--- Columns Retained ---")
print(keep_cols)

print("\n--- Shape After Column Selection ---")
print("Rows, Columns:", df_cols.shape)

print("\n--- Preview After Column Selection (Head) ---")
display(df_cols.head())


--- Columns Retained ---
['id', 'time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'type']

--- Shape After Column Selection ---
Rows, Columns: (9968, 8)

--- Preview After Column Selection (Head) ---


,id,time,latitude,longitude,depth,mag,place,type
0,ci41400888,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,"10 km S of Idyllwild, CA",earthquake
1,aka2026dpxhdt,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,"25 km N of Four Mile Road, Alaska",earthquake
2,ci41400872,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,"11 km S of Idyllwild, CA",earthquake
3,pr71508373,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake
4,aka2026dpwfxy,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,"55 km NNW of Petersville, Alaska",earthquake


In [13]:
# --- Handling Missing Coordinates (latitude/longitude) ---

before_rows = df_cols.shape[0]

missing_lat = df_cols["latitude"].isna().sum()
missing_lon = df_cols["longitude"].isna().sum()

print("--- Missing Values Before ---")
print("Missing latitude:", missing_lat)
print("Missing longitude:", missing_lon)
print("Rows before:", before_rows)

df_no_missing = df_cols.dropna(subset=["latitude", "longitude"]).copy()

after_rows = df_no_missing.shape[0]
print("\n--- After Dropping Missing Coordinates ---")
print("Rows after:", after_rows)
print("Removed rows:", before_rows - after_rows)

display(df_no_missing.head())

--- Missing Values Before ---
Missing latitude: 0
Missing longitude: 0
Rows before: 9968

--- After Dropping Missing Coordinates ---
Rows after: 9968
Removed rows: 0


,id,time,latitude,longitude,depth,mag,place,type
0,ci41400888,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,"10 km S of Idyllwild, CA",earthquake
1,aka2026dpxhdt,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,"25 km N of Four Mile Road, Alaska",earthquake
2,ci41400872,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,"11 km S of Idyllwild, CA",earthquake
3,pr71508373,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake
4,aka2026dpwfxy,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,"55 km NNW of Petersville, Alaska",earthquake


In [14]:
# --- Coordinate Range Validation ---

before_rows = df_no_missing.shape[0]

invalid_mask = ~(
    df_no_missing["latitude"].between(-90, 90) &
    df_no_missing["longitude"].between(-180, 180)
)

invalid_count = int(invalid_mask.sum())
print("--- Invalid Coordinate Rows ---")
print("Invalid rows found:", invalid_count)

# Optional: show a few invalid rows (if any)
if invalid_count > 0:
    print("\nSample invalid rows:")
    display(df_no_missing.loc[invalid_mask, ["latitude", "longitude"]].head())

df_valid_coords = df_no_missing.loc[~invalid_mask].copy()

after_rows = df_valid_coords.shape[0]
print("\n--- After Range Validation ---")
print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Removed rows:", before_rows - after_rows)

display(df_valid_coords.head())

--- Invalid Coordinate Rows ---
Invalid rows found: 0

--- After Range Validation ---
Rows before: 9968
Rows after: 9968
Removed rows: 0


,id,time,latitude,longitude,depth,mag,place,type
0,ci41400888,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,"10 km S of Idyllwild, CA",earthquake
1,aka2026dpxhdt,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,"25 km N of Four Mile Road, Alaska",earthquake
2,ci41400872,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,"11 km S of Idyllwild, CA",earthquake
3,pr71508373,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake
4,aka2026dpwfxy,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,"55 km NNW of Petersville, Alaska",earthquake


In [15]:
# --- Duplicate Removal (by unique earthquake id) ---

before_rows = df_valid_coords.shape[0]

# Count duplicates before removal
dup_count = int(df_valid_coords.duplicated(subset=["id"]).sum())
unique_ids_before = int(df_valid_coords["id"].nunique())

print("--- Duplicate Check (Before) ---")
print("Rows before:", before_rows)
print("Unique IDs before:", unique_ids_before)
print("Duplicate rows (by id):", dup_count)

# Remove duplicates
df_dedup = df_valid_coords.drop_duplicates(subset=["id"]).copy()

after_rows = df_dedup.shape[0]
unique_ids_after = int(df_dedup["id"].nunique())

print("\n--- After Removing Duplicates ---")
print("Rows after:", after_rows)
print("Unique IDs after:", unique_ids_after)
print("Removed rows:", before_rows - after_rows)

display(df_dedup.head())

--- Duplicate Check (Before) ---
Rows before: 9968
Unique IDs before: 9968
Duplicate rows (by id): 0

--- After Removing Duplicates ---
Rows after: 9968
Unique IDs after: 9968
Removed rows: 0


,id,time,latitude,longitude,depth,mag,place,type
0,ci41400888,2026-02-21T03:39:00.020Z,33.653000,-116.717500,15.40,0.49,"10 km S of Idyllwild, CA",earthquake
1,aka2026dpxhdt,2026-02-21T03:38:28.506Z,64.835000,-149.069000,12.30,1.00,"25 km N of Four Mile Road, Alaska",earthquake
2,ci41400872,2026-02-21T03:21:27.530Z,33.645667,-116.725500,15.26,1.58,"11 km S of Idyllwild, CA",earthquake
3,pr71508373,2026-02-21T03:09:43.060Z,19.192000,-66.487833,23.36,3.28,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake
4,aka2026dpwfxy,2026-02-21T03:06:52.985Z,62.930000,-151.281000,111.60,1.80,"55 km NNW of Petersville, Alaska",earthquake


In [16]:
# --- Data Type Conversion (time -> datetime, mag/depth -> numeric) ---

print("--- Dtypes BEFORE conversion ---")
print(df_dedup[["time", "mag", "depth"]].dtypes)

df_types = df_dedup.copy()

# Convert data types safely
df_types["time"] = pd.to_datetime(df_types["time"], errors="coerce")
df_types["mag"] = pd.to_numeric(df_types["mag"], errors="coerce")
df_types["depth"] = pd.to_numeric(df_types["depth"], errors="coerce")

print("\n--- Dtypes AFTER conversion ---")
print(df_types[["time", "mag", "depth"]].dtypes)

print("\n--- Missing Values AFTER conversion ---")
print(df_types[["time", "mag", "depth"]].isna().sum())

display(df_types.head())

--- Dtypes BEFORE conversion ---
time      object
mag      float64
depth    float64
dtype: object

--- Dtypes AFTER conversion ---
time     datetime64[ns, UTC]
mag                  float64
depth                float64
dtype: object

--- Missing Values AFTER conversion ---
time     0
mag      1
depth    0
dtype: int64


,id,time,latitude,longitude,depth,mag,place,type
0,ci41400888,2026-02-21 03:39:00.020000+00:00,33.653000,-116.717500,15.40,0.49,"10 km S of Idyllwild, CA",earthquake
1,aka2026dpxhdt,2026-02-21 03:38:28.506000+00:00,64.835000,-149.069000,12.30,1.00,"25 km N of Four Mile Road, Alaska",earthquake
2,ci41400872,2026-02-21 03:21:27.530000+00:00,33.645667,-116.725500,15.26,1.58,"11 km S of Idyllwild, CA",earthquake
3,pr71508373,2026-02-21 03:09:43.060000+00:00,19.192000,-66.487833,23.36,3.28,"80 km N of Tierras Nuevas Poniente, Puerto Rico",earthquake
4,aka2026dpwfxy,2026-02-21 03:06:52.985000+00:00,62.930000,-151.281000,111.60,1.80,"55 km NNW of Petersville, Alaska",earthquake
